# Sonar Pipelines: LDA+GBM, PCA+GBM, GBM baseline, and LdaBoost

This notebook provides a clean and reproducible comparison of several pipelines on the Sonar dataset. It mirrors the Iris pipeline structure while enforcing the same number of cross-validation folds across all models.

Sections:
- Dataset and preprocessing
- Reproducibility setup and constants
- LDA+GBM vs PCA+GBM (10-fold cross-validated)
- Baseline Gradient Boosting (10-fold cross-validated)
- LdaBoost (10-fold cross-validated)

In [1]:
"""
Sonar pipelines comparison notebook (cleaned and reproducible).

- Dataset: Sonar (UCI)
- Preprocessing: label encoding only (target)
- Models/Pipelines:
  1) LDA + Gradient Boosting
  2) PCA + Gradient Boosting
  3) Gradient Boosting baseline
  4) LdaBoost (custom boosting with LDA projections)
- Evaluation: Stratified K-Fold cross-validation accuracy (10-fold across all models)

This notebook is intended as supplementary material.
"""
# Reproducibility
import os
import random
import numpy as np

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Tuning constants
N_ESTIMATOR = 100
LEARNING_RATE = 0.1
MAX_DEPTH = 3

# Data and ML utilities
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_validate
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline

# Ensure local package import for LdaBoost
import sys
from pathlib import Path
PROJECT_ROOT = "../../"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
from LdaBoost import LdaBoost

# Load dataset
DATA_PATH = str(Path(PROJECT_ROOT) / "real_datasets" / "sonar" / "Data" / "sonar.csv")
# The CSV has no header; last column is the target (R/M)
num_feature_cols = 60
col_names = [f"f{i}" for i in range(num_feature_cols)] + ["target"]
data = pd.read_csv(DATA_PATH, header=None, names=col_names)

y = data["target"]
X = data.drop(columns="target").copy()

# Encode labels to integers
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Quick peek of the data
display(data.head())
print(f"Dataset shape: {data.shape}")


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,...,f51,f52,f53,f54,f55,f56,f57,f58,f59,target
0,0.0200,0.0371,0.0428,0.0207,0.0954,0.0986,0.1539,0.1601,0.3109,0.2111,...,0.0027,0.0065,0.0159,0.0072,0.0167,0.0180,0.0084,0.0090,0.0032,R
1,0.0453,0.0523,0.0843,0.0689,0.1183,0.2583,0.2156,0.3481,0.3337,0.2872,...,0.0084,0.0089,0.0048,0.0094,0.0191,0.0140,0.0049,0.0052,0.0044,R
2,0.0262,0.0582,0.1099,0.1083,0.0974,0.2280,0.2431,0.3771,0.5598,0.6194,...,0.0232,0.0166,0.0095,0.0180,0.0244,0.0316,0.0164,0.0095,0.0078,R
3,0.0100,0.0171,0.0623,0.0205,0.0205,0.0368,0.1098,0.1276,0.0598,0.1264,...,0.0121,0.0036,0.0150,0.0085,0.0073,0.0050,0.0044,0.0040,0.0117,R
4,0.0762,0.0666,0.0481,0.0394,0.0590,0.0649,0.1209,0.2467,0.3564,0.4459,...,0.0031,0.0054,0.0105,0.0110,0.0015,0.0072,0.0048,0.0107,0.0094,R


Dataset shape: (208, 61)


## Dataset and preprocessing

- Source: Sonar dataset (`sonar.csv`)
- Features: 60 continuous attributes
- Target: categorical label (`R`/`M`), label-encoded to integers
- Preprocessing: label encoding only
- CV strategy: 10-fold across all experiments


## LDA+GBM vs PCA+GBM (10-fold cross-validation)

We compare two pipelines with 10-fold cross-validation and `accuracy` scoring.


In [2]:
from dataclasses import dataclass, field
from typing import Any, Dict

@dataclass
class DuoBoostCV:
    seed: int = RANDOM_SEED
    _models: Dict[str, Any] = field(init=False, repr=False)

    def _build_models(self, n_features: int) -> Dict[str, Any]:
        half_components = max(1, n_features // 2)
        return {
            "PCA + GBM": Pipeline([
                ("pca", PCA(n_components=half_components, random_state=self.seed)),
                ("gb", GradientBoostingClassifier(
                    n_estimators=N_ESTIMATOR,
                    learning_rate=LEARNING_RATE,
                    max_depth=MAX_DEPTH,
                    random_state=self.seed,
                )),
            ]),
            "LDA + GBM": Pipeline([
                ("lda", LDA(n_components=None)),
                ("gb", GradientBoostingClassifier(
                    n_estimators=N_ESTIMATOR,
                    learning_rate=LEARNING_RATE,
                    max_depth=MAX_DEPTH,
                    random_state=self.seed,
                )),
            ]),
        }

    def fit(self, X_in, y_in, **cv_kwargs):
        X_arr = np.asarray(X_in)
        y_arr = np.asarray(y_in)
        cv_kwargs = cv_kwargs.copy()
        cv_kwargs.setdefault("scoring", "accuracy")
        cv_kwargs.setdefault("cv", 10)
        cv_kwargs.setdefault("n_jobs", -1)
        cv_kwargs.setdefault("return_train_score", True)

        self._models = self._build_models(X_arr.shape[1])
        scores: Dict[str, Dict[str, float]] = {}
        for name, model in self._models.items():
            cv_res = cross_validate(model, X_arr, y_arr, **cv_kwargs)
            metric_key = next(k for k in cv_res if k.startswith("test_"))
            metric_name = metric_key.replace("test_", "")
            scores[name] = {
                f"mean_{metric_name}": float(cv_res[metric_key].mean()),
                f"std_{metric_name}": float(cv_res[metric_key].std()),
            }
        self.results_ = pd.DataFrame(scores).T.sort_values(by=f"mean_{metric_name}", ascending=False)
        return self.results_

    def __call__(self, X_in, y_in, **cv_kwargs):
        return self.fit(X_in, y_in, **cv_kwargs)

trainer = DuoBoostCV(seed=RANDOM_SEED)
results_duo = trainer(X, y_encoded, cv=10, scoring="accuracy", n_jobs=-1, return_train_score=True)
display(results_duo.round(4))


,mean_score,std_score
PCA + GBM,0.6448,0.1322
LDA + GBM,0.6245,0.1502


## Baseline Gradient Boosting (10-fold cross-validation)

We evaluate a standalone Gradient Boosting classifier using the standardized constants and 10-fold CV.


In [3]:
gb_clf = GradientBoostingClassifier(
    n_estimators=N_ESTIMATOR,
    learning_rate=LEARNING_RATE,
    max_depth=MAX_DEPTH,
    random_state=RANDOM_SEED,
)

cv_scores = cross_validate(
    gb_clf,
    X,
    y_encoded,
    cv=10,
    scoring="accuracy",
    n_jobs=-1,
    return_train_score=True,
)

test_key = next(k for k in cv_scores if k.startswith("test_"))
metric_name = test_key.replace("test_", "")
mean_acc = float(cv_scores[test_key].mean())
std_acc = float(cv_scores[test_key].std())
print(f"GBM {metric_name}: {mean_acc:.4f} ± {std_acc:.4f}")


GBM score: 0.6936 ± 0.1728


## LdaBoost (10-fold cross-validation)

We evaluate the custom `LdaBoost` classifier imported from the local package, using the same constants and folds.


In [4]:
ldaboost = LdaBoost(
    n_estimators=N_ESTIMATOR,
    learning_rate=LEARNING_RATE,
    max_depth=MAX_DEPTH,
    random_state=RANDOM_SEED,
)

accs = ldaboost.cross_validate(X=np.asarray(X), y=np.asarray(y_encoded), cv=10)
print(f"LdaBoost accuracy: {np.mean(accs):.4f} ± {np.std(accs):.4f}")


LdaBoost accuracy: 0.7214 ± 0.0751
